<a href="https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/paper/hcf-optimum-launch-power-reproduction/dnanf_fig2_hcf_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fig. 2 HCF / G.657.A2 처리량 — 선택형 물리 기반 Colab

이 노트북은 `FIBER_SELECTION` 값으로 **HCF**, **G.657.A2**, 또는 **ALL**을 선택해 C-band 처리량을 계산합니다.

```python
FIBER_SELECTION = "HCF"      # HCF만 계산
FIBER_SELECTION = "G657A2"  # G.657.A2만 계산
FIBER_SELECTION = "ALL"      # 두 결과를 한 그래프에서 비교
```

각 C-band 채널에 대해 다음 모델을 직접 계산합니다.

$$
\mathrm{SNR}_i = \frac{P_i}{P_{\mathrm{ASE},i}+\eta_iP_i^3+P_{\mathrm{IMI},i}+P_{\mathrm{TRN},i}},
\qquad
T_i=2R\log_2(1+\mathrm{SNR}_i).
$$

- **공통 입력:** 채널 수, baud rate, 채널 간격, 증폭기 NF, 송수신기 SNR
- **HCF 전용 입력:** DNANF 형상, \(f_1/f_2\) 보정, anti-resonant 손실, HCF \(\gamma\), IMI, 1×200 km
- **G.657.A2 전용 입력:** 감쇠, 분산/기울기, \(A_{\rm eff}\), 실리카 \(n_2\), 선택적 굽힘 손실, 2×100 km

G.657.A2를 선택하면 HCF 전용 형상·\(f_1/f_2\)·IMI 계산은 실행되지 않습니다. `ALL`은 각 광섬유를 독립 계산한 뒤 같은 launch-power 축과 throughput 축에 표시합니다.

> G.657.A2 기본값은 특정 제조사 제품 보증값이 아니라 대표적인 C-band engineering 값입니다. 정밀 비교 시 `G657A2Parameters`를 실제 제품 데이터시트로 교체하십시오.

## HCF 논문 곡선 검증

HCF가 포함된 경우에만 논문 Fig. 2에서 추출한 8개 별표와 RMSE/MAPE를 표시합니다. 검증점은 모델 입력이나 보정에 사용되지 않습니다.


In [1]:
%pip -q install numpy scipy matplotlib


## 실행 및 입력값 수정 방법

1. 설치 셀을 실행합니다.
2. 모델 셀 맨 위의 `FIBER_SELECTION`을 `"HCF"`, `"G657A2"`, `"G.657.A2"`, 또는 `"ALL"`로 설정합니다.
3. 공통 조건은 `TransmissionSystem`, HCF 전용 값은 `HCFParameters`와 `DNANFGeometry`, G.657.A2 전용 값은 `G657A2Parameters`에서 수정합니다.
4. 모델 셀을 실행하면 선택에 맞는 요약값과 그래프가 출력됩니다.

기본 비교에서는 총 전송거리 200 km를 맞추고, HCF는 논문 조건인 1×200 km, G.657.A2는 손실을 고려한 2×100 km로 설정했습니다. 동일 span 구성의 순수 섬유 비교가 필요하면 두 데이터클래스의 `span_length_km`와 `n_spans`를 같게 설정하십시오.

출력 그림은 선택에 따라 `fig2_hcf_throughput.png`, `fig2_g657a2_throughput.png`, 또는 `fig2_all_throughput.png`로 저장됩니다.


In [ ]:
"""Selectable HCF / G.657.A2 C-band throughput model.

Set FIBER_SELECTION to "HCF", "G657A2", or "ALL".  Each fibre type owns
only its applicable physical inputs; HCF geometry and IMI are never requested
for G.657.A2.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares


# User switch: "HCF", "G657A2", or "ALL".
FIBER_SELECTION = "ALL"

C0 = 299_792_458.0
H_PLANCK = 6.626_070_15e-34
U01 = 2.4048255577
DB_PER_NEPER = 10.0 / np.log(10.0)


# ---------------------------------------------------------------------------
# 1) Common transmission-system inputs
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class TransmissionSystem:
    """Inputs shared by both fibres so the throughput curves are comparable."""

    n_channels: int = 29
    symbol_rate_gbd: float = 140.0
    channel_spacing_ghz: float = 150.0
    c_band_min_nm: float = 1530.0
    c_band_max_nm: float = 1565.0
    amplifier_noise_figure_db: float = 5.0
    transceiver_snr_db: float = 20.0

    @property
    def symbol_rate_hz(self) -> float:
        return self.symbol_rate_gbd * 1e9

    @property
    def channel_spacing_hz(self) -> float:
        return self.channel_spacing_ghz * 1e9


# ---------------------------------------------------------------------------
# 2) HCF-only physical inputs and models
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class DNANFGeometry:
    core_radius_um: float = 14.75
    membrane_thickness_um: float = 0.50
    outer_tube_diameter_um: float = 31.05
    tube_count: int = 5
    nesting_order: int = 2

    @property
    def core_radius_m(self) -> float:
        return self.core_radius_um * 1e-6

    @property
    def membrane_thickness_m(self) -> float:
        return self.membrane_thickness_um * 1e-6

    @property
    def perimeter_gap_m(self) -> float:
        theta = np.pi / self.tube_count
        return (
            2.0 * self.core_radius_m * np.sin(theta)
            - self.outer_tube_diameter_um * 1e-6 * (1.0 - np.sin(theta))
        )


@dataclass(frozen=True)
class HCFParameters:
    """Parameters used only when the selected fibre is HCF."""

    geometry: DNANFGeometry = DNANFGeometry()
    span_length_km: float = 200.0
    n_spans: int = 1
    nonlinear_coefficient_per_w_km: float = 5e-4
    imi_coefficient_db_per_km: float = -52.0

    @property
    def total_length_km(self) -> float:
        return self.span_length_km * self.n_spans


def silica_index_sellmeier(wavelength_m: np.ndarray) -> np.ndarray:
    wavelength_um = np.asarray(wavelength_m, dtype=float) * 1e6
    wavelength_um_sq = wavelength_um**2
    b = np.array([0.6961663, 0.4079426, 0.8974794])
    c_um = np.array([0.0684043, 0.1162414, 9.896161])
    n_sq = np.ones_like(wavelength_um_sq)
    for bi, ci in zip(b, c_um):
        n_sq += bi * wavelength_um_sq / (wavelength_um_sq - ci**2)
    return np.sqrt(n_sq)


def hasan_radius_coefficients(geometry: DNANFGeometry) -> tuple[float, float]:
    radius_to_gap = geometry.core_radius_m / geometry.perimeter_gap_m
    n = geometry.tube_count
    nesting = geometry.nesting_order
    a0, a1 = 0.097041, 1.095
    b0, b1, b2, b3 = 0.76246, 0.007584, 0.002, 0.012
    f1 = a1 * np.exp(a0 / radius_to_gap)
    f2 = (
        b1 * n * np.exp(b0 / radius_to_gap)
        - b2 * n
        + b3
        + 0.0045 * np.exp(-4.1589 / (nesting * radius_to_gap))
    )
    return float(f1), float(f2)


def effective_index(
    wavelength_m: np.ndarray,
    geometry: DNANFGeometry,
    f1: float,
    f2: float,
) -> np.ndarray:
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    radius = geometry.core_radius_m
    wall = geometry.membrane_thickness_m
    effective_radius = f1 * radius * (
        1.0 - f2 * wavelength_m**2 / (radius * wall)
    )
    return 1.0 - 0.125 * (
        U01 * wavelength_m / (np.pi * effective_radius)
    ) ** 2


def hcf_chromatic_dispersion_ps_nm_km(
    wavelength_m: np.ndarray,
    geometry: DNANFGeometry,
    f1: float,
    f2: float,
    derivative_step_nm: float = 0.20,
) -> np.ndarray:
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    step = derivative_step_nm * 1e-9

    def n_eff(offset: float) -> np.ndarray:
        return effective_index(wavelength_m + offset, geometry, f1, f2)

    second_derivative = (
        -n_eff(2.0 * step)
        + 16.0 * n_eff(step)
        - 30.0 * n_eff(0.0)
        + 16.0 * n_eff(-step)
        - n_eff(-2.0 * step)
    ) / (12.0 * step**2)
    dispersion_si = -(wavelength_m / C0) * second_derivative
    return dispersion_si / 1e-6


def calibrate_hcf_dispersion(geometry: DNANFGeometry) -> tuple[float, float]:
    reference_nm = np.array([1310.0, 1550.0])
    reference_d = np.array([2.17, 3.20])
    initial = np.array(hasan_radius_coefficients(geometry))

    def residual(parameters: np.ndarray) -> np.ndarray:
        calculated = hcf_chromatic_dispersion_ps_nm_km(
            reference_nm * 1e-9, geometry, parameters[0], parameters[1]
        )
        return calculated - reference_d

    result = least_squares(
        residual,
        x0=initial,
        bounds=(np.array([0.5, -0.5]), np.array([2.5, 1.0])),
        x_scale="jac",
        diff_step=1e-3,
        xtol=1e-13,
        ftol=1e-13,
        gtol=1e-13,
    )
    if not result.success or np.max(np.abs(result.fun)) > 1e-3:
        raise RuntimeError(f"HCF dispersion calibration failed: {result.fun}")
    return float(result.x[0]), float(result.x[1])


def capillary_bouncing_ray_loss_db_km(
    wavelength_m: np.ndarray,
    geometry: DNANFGeometry,
) -> np.ndarray:
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    k0 = 2.0 * np.pi / wavelength_m
    radius = geometry.core_radius_m
    wall = geometry.membrane_thickness_m
    kappa = U01 / radius
    silica_n = silica_index_sellmeier(wavelength_m)
    sigma = k0 * np.sqrt(silica_n**2 - 1.0)
    phase = sigma * wall
    te_denominator = 4.0 * np.cos(phase) ** 2 + (
        kappa / sigma + sigma / kappa
    ) ** 2 * np.sin(phase) ** 2
    tm_denominator = 4.0 * np.cos(phase) ** 2 + (
        silica_n**2 * kappa / sigma + sigma / (silica_n**2 * kappa)
    ) ** 2 * np.sin(phase) ** 2
    alpha_te_per_m = 2.0 * U01 / (radius**2 * k0 * te_denominator)
    alpha_tm_per_m = 2.0 * U01 / (radius**2 * k0 * tm_denominator)
    return (
        0.5
        * (alpha_te_per_m + alpha_tm_per_m)
        * DB_PER_NEPER
        * 1000.0
    )


def hcf_attenuation_db_km(
    wavelength_m: np.ndarray,
    geometry: DNANFGeometry,
) -> np.ndarray:
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    reference_nm = np.array([1550.0, 1310.0])
    reference_alpha = np.array([0.075, 0.120])
    lambda_ref_m = reference_nm[0] * 1e-9
    br_ref = capillary_bouncing_ray_loss_db_km(reference_nm * 1e-9, geometry)
    basis_reference = np.column_stack(
        [
            (lambda_ref_m / (reference_nm * 1e-9)) ** 3,
            (br_ref / br_ref[0]) ** 2,
        ]
    )
    a_surface, a_nested = np.linalg.solve(basis_reference, reference_alpha)
    if a_surface < 0.0 or a_nested < 0.0:
        raise RuntimeError("Calibrated HCF attenuation terms must be non-negative.")
    br = capillary_bouncing_ray_loss_db_km(wavelength_m, geometry)
    return (
        a_surface * (lambda_ref_m / wavelength_m) ** 3
        + a_nested * (br / br_ref[0]) ** 2
    )


# ---------------------------------------------------------------------------
# 3) G.657.A2-only physical inputs and models
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class G657A2Parameters:
    """Representative engineering inputs for a straight G.657.A2 link.

    Replace these defaults with the selected manufacturer's measured data.
    HCF geometry, f1/f2 and IMI do not exist in this class.
    """

    span_length_km: float = 100.0
    n_spans: int = 2
    attenuation_1550_db_per_km: float = 0.21
    attenuation_slope_db_per_km_nm: float = 2.0e-4
    dispersion_1550_ps_nm_km: float = 17.0
    dispersion_slope_ps_nm2_km: float = 0.058
    effective_area_um2: float = 80.0
    nonlinear_index_m2_per_w: float = 2.6e-20
    additional_bend_loss_db_per_km: float = 0.0

    @property
    def total_length_km(self) -> float:
        return self.span_length_km * self.n_spans


def g657a2_attenuation_db_km(
    wavelength_m: np.ndarray,
    parameters: G657A2Parameters,
) -> np.ndarray:
    """Representative C-band attenuation plus optional distributed bend loss."""
    wavelength_nm = np.asarray(wavelength_m, dtype=float) * 1e9
    return (
        parameters.attenuation_1550_db_per_km
        + parameters.attenuation_slope_db_per_km_nm
        * np.abs(wavelength_nm - 1550.0)
        + parameters.additional_bend_loss_db_per_km
    )


def g657a2_dispersion_ps_nm_km(
    wavelength_m: np.ndarray,
    parameters: G657A2Parameters,
) -> np.ndarray:
    wavelength_nm = np.asarray(wavelength_m, dtype=float) * 1e9
    return (
        parameters.dispersion_1550_ps_nm_km
        + parameters.dispersion_slope_ps_nm2_km * (wavelength_nm - 1550.0)
    )


def g657a2_gamma_per_w_km(
    wavelength_m: np.ndarray,
    parameters: G657A2Parameters,
) -> np.ndarray:
    effective_area_m2 = parameters.effective_area_um2 * 1e-12
    return (
        2.0
        * np.pi
        * parameters.nonlinear_index_m2_per_w
        / (np.asarray(wavelength_m, dtype=float) * effective_area_m2)
        * 1e3
    )


# ---------------------------------------------------------------------------
# 4) Selection layer and shared GN / ASE / throughput calculation
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class FiberProfile:
    name: str
    span_length_km: float
    n_spans: int
    attenuation_db_per_km: np.ndarray
    dispersion_ps_nm_km: np.ndarray
    gamma_per_w_km: np.ndarray
    imi_coefficient_db_per_km: float | None
    metadata: dict[str, float]

    @property
    def total_length_km(self) -> float:
        return self.span_length_km * self.n_spans


def normalize_fiber_selection(selection: str) -> str:
    normalized = selection.upper().replace(".", "").replace("_", "")
    aliases = {
        "HCF": "HCF",
        "G657A2": "G657A2",
        "ALL": "ALL",
    }
    if normalized not in aliases:
        raise ValueError(
            'FIBER_SELECTION must be "HCF", "G657A2" (or "G.657.A2"), or "ALL".'
        )
    return aliases[normalized]


def c_band_channel_grid(
    system: TransmissionSystem,
) -> tuple[np.ndarray, np.ndarray]:
    lower_frequency_hz = C0 / (system.c_band_max_nm * 1e-9)
    upper_frequency_hz = C0 / (system.c_band_min_nm * 1e-9)
    center_frequency_hz = 0.5 * (lower_frequency_hz + upper_frequency_hz)
    offsets = np.arange(system.n_channels) - 0.5 * (system.n_channels - 1)
    frequency_hz = (
        center_frequency_hz + offsets * system.channel_spacing_hz
    )
    wavelength_m = C0 / frequency_hz
    if (
        frequency_hz[0] < lower_frequency_hz
        or frequency_hz[-1] > upper_frequency_hz
    ):
        raise ValueError("The selected channel grid does not fit inside the C-band.")
    return frequency_hz, wavelength_m


def build_fiber_profile(
    fiber_type: str,
    wavelength_m: np.ndarray,
) -> FiberProfile:
    """Create only the parameter set needed by the selected fibre type."""
    selected = normalize_fiber_selection(fiber_type)

    match selected:
        case "HCF":
            parameters = HCFParameters()
            geometry = parameters.geometry
            f1, f2 = calibrate_hcf_dispersion(geometry)
            attenuation = hcf_attenuation_db_km(wavelength_m, geometry)
            dispersion = hcf_chromatic_dispersion_ps_nm_km(
                wavelength_m, geometry, f1, f2
            )
            gamma = np.full_like(
                wavelength_m,
                parameters.nonlinear_coefficient_per_w_km,
                dtype=float,
            )
            return FiberProfile(
                name="HCF",
                span_length_km=parameters.span_length_km,
                n_spans=parameters.n_spans,
                attenuation_db_per_km=attenuation,
                dispersion_ps_nm_km=dispersion,
                gamma_per_w_km=gamma,
                imi_coefficient_db_per_km=parameters.imi_coefficient_db_per_km,
                metadata={"f1": f1, "f2": f2},
            )

        case "G657A2":
            parameters = G657A2Parameters()
            attenuation = g657a2_attenuation_db_km(wavelength_m, parameters)
            dispersion = g657a2_dispersion_ps_nm_km(wavelength_m, parameters)
            gamma = g657a2_gamma_per_w_km(wavelength_m, parameters)
            return FiberProfile(
                name="G.657.A2",
                span_length_km=parameters.span_length_km,
                n_spans=parameters.n_spans,
                attenuation_db_per_km=attenuation,
                dispersion_ps_nm_km=dispersion,
                gamma_per_w_km=gamma,
                imi_coefficient_db_per_km=None,
                metadata={"effective_area_um2": parameters.effective_area_um2},
            )

        case _:
            raise ValueError("ALL is a plotting selection, not one fibre profile.")


def dispersion_to_beta2_s2_per_km(
    dispersion_ps_nm_km: np.ndarray,
    wavelength_m: np.ndarray,
) -> np.ndarray:
    dispersion_si = np.asarray(dispersion_ps_nm_km) * 1e-6
    beta2_s2_per_m = -(
        np.asarray(wavelength_m) ** 2 / (2.0 * np.pi * C0)
    ) * dispersion_si
    return beta2_s2_per_m * 1000.0


def closed_form_gn_eta(
    beta2_s2_per_km: np.ndarray,
    attenuation_db_per_km: np.ndarray,
    gamma_per_w_km: np.ndarray,
    profile: FiberProfile,
    system: TransmissionSystem,
) -> np.ndarray:
    beta2 = np.abs(np.asarray(beta2_s2_per_km, dtype=float))
    alpha_field_per_km = (
        np.asarray(attenuation_db_per_km, dtype=float) / 8.685889638
    )
    gamma = np.asarray(gamma_per_w_km, dtype=float)
    rate = system.symbol_rate_hz
    spacing = system.channel_spacing_hz
    asinh_argument = (
        np.pi**2
        * beta2
        * rate**2
        / (4.0 * alpha_field_per_km)
        * system.n_channels ** (2.0 * rate / spacing)
    )
    return (
        profile.n_spans
        * 4.0
        * gamma**2
        / (27.0 * np.pi * beta2 * alpha_field_per_km * rate**2)
        * np.arcsinh(asinh_argument)
    )


def per_channel_ase_w(
    frequency_hz: np.ndarray,
    profile: FiberProfile,
    system: TransmissionSystem,
) -> np.ndarray:
    span_gain = 10.0 ** (
        profile.attenuation_db_per_km * profile.span_length_km / 10.0
    )
    noise_factor = 10.0 ** (system.amplifier_noise_figure_db / 10.0)
    return (
        profile.n_spans
        * H_PLANCK
        * np.asarray(frequency_hz)
        * noise_factor
        * system.symbol_rate_hz
        * (span_gain - 1.0)
    )


def simulate_fiber_throughput(
    fiber_type: str,
    system: TransmissionSystem,
    launch_dbm: np.ndarray,
) -> dict[str, Any]:
    launch_dbm = np.asarray(launch_dbm, dtype=float)
    frequency_hz, wavelength_m = c_band_channel_grid(system)
    profile = build_fiber_profile(fiber_type, wavelength_m)
    beta2 = dispersion_to_beta2_s2_per_km(
        profile.dispersion_ps_nm_km, wavelength_m
    )
    eta = closed_form_gn_eta(
        beta2,
        profile.attenuation_db_per_km,
        profile.gamma_per_w_km,
        profile,
        system,
    )
    ase_w = per_channel_ase_w(frequency_hz, profile, system)

    launch_w = 10.0 ** ((launch_dbm - 30.0) / 10.0)
    signal = launch_w[:, None]
    nonlinear = eta[None, :] * signal**3
    if profile.imi_coefficient_db_per_km is None:
        imi = np.zeros_like(signal)
    else:
        imi_ratio_per_km = 10.0 ** (
            profile.imi_coefficient_db_per_km / 10.0
        )
        imi = signal * imi_ratio_per_km * profile.total_length_km
    transceiver = signal / (10.0 ** (system.transceiver_snr_db / 10.0))
    snr = signal / (ase_w[None, :] + nonlinear + imi + transceiver)
    throughput_per_channel_bps = (
        2.0 * system.symbol_rate_hz * np.log2(1.0 + snr)
    )
    total_throughput_tbps = throughput_per_channel_bps.sum(axis=1) / 1e12
    optimum_index = int(np.argmax(total_throughput_tbps))

    if not np.all(np.isfinite(total_throughput_tbps)):
        raise RuntimeError(f"{profile.name}: non-finite throughput was calculated.")
    maximum = float(total_throughput_tbps[optimum_index])
    if maximum <= 0.0:
        raise RuntimeError(f"{profile.name}: throughput must be positive.")

    return {
        "fiber_name": profile.name,
        "profile": profile,
        "launch_dbm": launch_dbm,
        "frequency_hz": frequency_hz,
        "wavelength_m": wavelength_m,
        "beta2_s2_km": beta2,
        "eta_per_w2": eta,
        "ase_w": ase_w,
        "snr_linear": snr,
        "throughput_per_channel_bps": throughput_per_channel_bps,
        "total_throughput_tbps": total_throughput_tbps,
        "optimum_index": optimum_index,
        "optimum_launch_dbm": float(launch_dbm[optimum_index]),
        "maximum_throughput_tbps": float(total_throughput_tbps[optimum_index]),
    }


PAPER_REFERENCE_LAUNCH_DBM = np.array(
    [-35.0, -20.0, -10.0, 0.0, 10.0, 22.0, 40.0, 48.0]
)
PAPER_REFERENCE_THROUGHPUT_TBPS = np.array(
    [
        1.87292462,
        21.32203450,
        41.62572367,
        50.95120740,
        52.49881133,
        52.67046429,
        42.03103403,
        10.44268748,
    ]
)


def compare_hcf_with_paper(result: dict[str, Any]) -> dict[str, Any]:
    model_at_reference = np.interp(
        PAPER_REFERENCE_LAUNCH_DBM,
        result["launch_dbm"],
        result["total_throughput_tbps"],
    )
    error = model_at_reference - PAPER_REFERENCE_THROUGHPUT_TBPS
    relative_error = 100.0 * error / PAPER_REFERENCE_THROUGHPUT_TBPS
    return {
        "model_tbps": model_at_reference,
        "relative_error_percent": relative_error,
        "rmse_tbps": float(np.sqrt(np.mean(error**2))),
        "mape_percent": float(np.mean(np.abs(relative_error))),
    }


def selected_fiber_names(selection: str) -> list[str]:
    selected = normalize_fiber_selection(selection)
    match selected:
        case "HCF":
            return ["HCF"]
        case "G657A2":
            return ["G657A2"]
        case "ALL":
            return ["HCF", "G657A2"]
        case _:
            raise AssertionError("Unreachable selection.")


def plot_results(
    results: dict[str, dict[str, Any]],
    selection: str,
    output_path: Path | str,
) -> plt.Figure:
    styles = {
        "HCF": {"color": "#e66100", "label": "HCF (1 × 200 km)"},
        "G657A2": {"color": "#1666b1", "label": "G.657.A2 (2 × 100 km)"},
    }
    fig, ax = plt.subplots(figsize=(8.6, 5.8), constrained_layout=True)
    positive_values: list[np.ndarray] = []

    for key, result in results.items():
        style = styles[key]
        launch_dbm = result["launch_dbm"]
        throughput = result["total_throughput_tbps"]
        optimum_index = result["optimum_index"]
        positive_values.append(throughput[throughput > 0.0])
        ax.plot(
            launch_dbm,
            throughput,
            color=style["color"],
            lw=2.6,
            label=style["label"],
        )
        ax.plot(
            launch_dbm[optimum_index],
            throughput[optimum_index],
            marker="s",
            ms=7,
            color=style["color"],
            markeredgecolor="white",
            markeredgewidth=0.8,
            zorder=5,
        )
        ax.annotate(
            f"{result['fiber_name']}: {result['optimum_launch_dbm']:.1f} dBm, "
            f"{result['maximum_throughput_tbps']:.2f} Tb/s",
            xy=(launch_dbm[optimum_index], throughput[optimum_index]),
            xytext=(8, 10 if key == "HCF" else -22),
            textcoords="offset points",
            fontsize=9.2,
            color=style["color"],
        )

    if "HCF" in results:
        ax.scatter(
            PAPER_REFERENCE_LAUNCH_DBM,
            PAPER_REFERENCE_THROUGHPUT_TBPS,
            marker="*",
            s=105,
            color="#4d4d4d",
            edgecolors="white",
            linewidths=0.7,
            label="Paper Fig. 2 HCF reference",
            zorder=6,
        )

    all_positive = np.concatenate(positive_values)
    y_min = max(1e-3, 10.0 ** np.floor(np.log10(all_positive.min())))
    y_max = max(70.0, 1.25 * all_positive.max())
    ax.set_yscale("log")
    ax.set_xlim(-40.0, 50.0)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("Launch power per channel (dBm)", fontsize=12)
    ax.set_ylabel("C-band throughput (Tb/s)", fontsize=12)
    title_selection = normalize_fiber_selection(selection)
    ax.set_title(
        f"HCF / G.657.A2 throughput comparison — {title_selection}",
        fontsize=13,
    )
    ax.grid(True, which="major", color="#8a8a8a", alpha=0.40, linewidth=0.8)
    ax.grid(True, which="minor", axis="y", color="#b5b5b5", alpha=0.16)
    ax.legend(loc="best", frameon=True, framealpha=0.95, fontsize=9.3)
    fig.savefig(Path(output_path), dpi=220, bbox_inches="tight")
    return fig


def print_summary(
    results: dict[str, dict[str, Any]],
    system: TransmissionSystem,
) -> None:
    print(
        "Channels / baud rate / spacing : "
        f"{system.n_channels} / {system.symbol_rate_gbd:.0f} GBd / "
        f"{system.channel_spacing_ghz:.0f} GHz"
    )
    for key, result in results.items():
        profile: FiberProfile = result["profile"]
        wavelength_nm = result["wavelength_m"] * 1e9
        print(f"\n=== {profile.name} calculation ===")
        print(
            f"Link spans / total length       : {profile.n_spans} × "
            f"{profile.span_length_km:.1f} km / {profile.total_length_km:.1f} km"
        )
        print(
            f"C-band wavelengths              : {wavelength_nm.min():.2f}–"
            f"{wavelength_nm.max():.2f} nm"
        )
        print(
            f"Attenuation                     : "
            f"{profile.attenuation_db_per_km.min():.4f}–"
            f"{profile.attenuation_db_per_km.max():.4f} dB/km"
        )
        print(
            f"Dispersion                      : "
            f"{profile.dispersion_ps_nm_km.min():.4f}–"
            f"{profile.dispersion_ps_nm_km.max():.4f} ps/(nm·km)"
        )
        print(
            f"Nonlinear coefficient gamma     : "
            f"{profile.gamma_per_w_km.min():.4g}–"
            f"{profile.gamma_per_w_km.max():.4g} 1/(W·km)"
        )
        print(
            f"GN eta                          : {result['eta_per_w2'].min():.3e}–"
            f"{result['eta_per_w2'].max():.3e} 1/W²"
        )
        print(
            f"Mean ASE/channel                : "
            f"{10.0 * np.log10(result['ase_w'].mean()) + 30.0:.2f} dBm"
        )
        print(
            f"Maximum throughput              : "
            f"{result['maximum_throughput_tbps']:.3f} Tb/s"
        )
        print(
            f"Optimum launch power            : "
            f"{result['optimum_launch_dbm']:.2f} dBm/channel"
        )
        if key == "HCF":
            comparison = compare_hcf_with_paper(result)
            print(
                f"HCF Fig. 2 RMSE / MAPE           : "
                f"{comparison['rmse_tbps']:.4f} Tb/s / "
                f"{comparison['mape_percent']:.3f}%"
            )
            print(
                f"HCF calibrated f1 / f2           : "
                f"{profile.metadata['f1']:.6f} / {profile.metadata['f2']:.6f}"
            )


def validate_results(results: dict[str, dict[str, Any]]) -> None:
    for result in results.values():
        assert np.all(np.isfinite(result["total_throughput_tbps"]))
        assert result["maximum_throughput_tbps"] > 0.0
        assert -40.0 <= result["optimum_launch_dbm"] <= 50.0
    if "HCF" in results:
        assert 50.0 < results["HCF"]["maximum_throughput_tbps"] < 55.0
        assert 15.0 < results["HCF"]["optimum_launch_dbm"] < 30.0
    if "G657A2" in results:
        assert 0.5 < results["G657A2"]["profile"].gamma_per_w_km.mean() < 2.5
    print("Validation: PASS")


def main(
    selection: str = FIBER_SELECTION,
) -> tuple[dict[str, dict[str, Any]], plt.Figure]:
    system = TransmissionSystem()
    launch_dbm = np.linspace(-40.0, 50.0, 1201)
    fiber_names = selected_fiber_names(selection)
    results = {
        fiber_name: simulate_fiber_throughput(
            fiber_name, system, launch_dbm
        )
        for fiber_name in fiber_names
    }
    output_name = (
        f"fig2_{normalize_fiber_selection(selection).lower()}_throughput.png"
    )
    figure = plot_results(results, selection, Path(output_name))
    print_summary(results, system)
    validate_results(results)
    plt.show()
    return results, figure


if __name__ == "__main__":
    results, figure = main()


## 모델 참고문헌과 입력값 주의사항

1. R. Sohanpal *et al.*, “On the Optimum Energy-per-bit Launch Power in Coherent Hollow-core Fibre Transmission Systems,” 2026.
2. P. Poggiolini and F. Poletti, “Opportunities and Challenges for Long-Distance Transmission in Hollow-Core Fibres,” *J. Lightwave Technol.*, 40(6), 1605–1616 (2022), DOI: 10.1109/JLT.2021.3140114.
3. M. I. Hasan *et al.*, “Analytical model of the effective modal refractive index of the LP01 mode of non-ideal hollow-core anti-resonant fibers,” arXiv:1708.06879.
4. ITU-T Recommendation G.657 (08/24), “Characteristics of a bending-loss insensitive single-mode optical fibre and cable.”

G.657.A2 Recommendation은 허용 규격을 정의하므로 제조사별 실제 감쇠, MFD/유효면적, 분산 및 굽힘 손실은 다를 수 있습니다. 이 노트북의 G.657.A2 값은 코드 구조와 비교 계산을 위한 대표값이며, 제품 선정 또는 설계 확정에는 해당 제조사의 실측 데이터시트를 사용해야 합니다.
